In [ ]:
"""
Data Processing Module

Dependencies:
     - pandas: Data manipulation
     - os: File path operations
     - data_production: Custom module for running the data pipeline

Global Variables:
     WRK_DIR (str): Working directory path for data files
"""

import pandas as pd 
import os 
import data_production
WRK_DIR = '/mnt/shared/cosmosuk_data'

sites = pd.read_csv('/home/vm/Desktop/cosmos-uk_sitemetadata_2013-2023.csv')
land_cover_to_plant_type = {
    "Broadleaf woodland": "tree",
    "Coniferous woodland": "tree",
    "Coniferous forest": "tree",
    "Arable and horticulture": "crop",
    "Arable and horticulture (previously improved grassland)": "crop",
    "Orchard": "crop",
    "Grassland": "grass",
    "Improved grassland": "grass",
    "Heather grassland": "grass",
    "Acid grassland": "grass",
    "Calcareous grassland": "grass",
    "Heather": "shrub",
}

sites["plant_type"] = sites["LAND_COVER"].astype(str).str.strip().map(land_cover_to_plant_type)
sites[["LAND_COVER", "plant_type"]].drop_duplicates().sort_values(["plant_type", "LAND_COVER"])

for ct in range(sites["SITE_ID"].nunique()):
    site = sites.SITE_ID[sites.SITE_ID == sites.SITE_ID.values[ct]].values[0]
    lat = sites.LATITUDE.values[ct]
    lon = sites.LONGITUDE.values[ct]
    site_type = sites['plant_type'].values[ct]

    try:
        data_production.run_pipeline(
            wrk_dir=WRK_DIR,
            site=site,
            lat=lat,
            lon=lon,
            start_date="2020-01-01",
            end_date="2025-12-31",
            site_type = site_type
        )
    except Exception as exc:
        print(f"data_production failed for site {site} at index {ct}: {exc}")
        continue

[SUCCESS] Step 0: Runtime context initialized for site=ALIC1 with AOI bounds W=-0.8596641826391378, S=51.15265268882501, E=-0.8567998173608622, N=51.15444931117499.
Authenticated using refresh token.
0:00:00 Job 'cdse-j-26040314204649f7b08a3ac035b2e1ba': send 'start'
0:00:16 Job 'cdse-j-26040314204649f7b08a3ac035b2e1ba': created (progress 0%)
0:00:21 Job 'cdse-j-26040314204649f7b08a3ac035b2e1ba': created (progress 0%)
0:00:28 Job 'cdse-j-26040314204649f7b08a3ac035b2e1ba': running (progress N/A)
0:00:36 Job 'cdse-j-26040314204649f7b08a3ac035b2e1ba': running (progress N/A)


In [ ]:
from dataclasses import dataclass, field, fields, replace
from typing import Dict, List, Optional, Sequence
import numpy as np
import pandas as pd
import xarray as xr 
import matplotlib.pyplot as plt
import sys
from pathlib import Path
src_path = Path("/home/vm/Desktop/soil_moisture_model/src")
if str(src_path) not in sys.path : sys.path.insert(0, str(src_path))
from soil_moisture_model.soil_moisture_model import SoilWaterParams, simulate_soil_water_balance, params_as_dict

In [ ]:
_D = pd.read_csv(
    "/home/vm/Desktop/cosmos-api-download-20260403140459/COSMOS_UK_RISEH_1D_202201010000_202512310000.csv",
    header=5,
    skiprows=[6, 7]
)
_D = _D.rename(columns={"parameter-id": "date"})
_D["date"] = pd.to_datetime(_D["date"], errors="coerce")
_D

In [ ]:
sites.columns

In [ ]:
site_meta = sites_df.loc[sites_df["SITE_ID"] == site_name].iloc[0]
elevation = float(site_meta["ALTITUDE"])
plant_type = site_meta["plant_type"] if "plant_type" in site_meta.index else np.nan

lai_to_biomass_multiplier = {
    "tree": 600.0,
    "shrub": 100.0,
    "grass": 100.0,
    "crop": 200.0,
}.get(str(plant_type).strip().lower(), 100.0)

biomass = D_site.modis_lai[:, 0, 0].data * lai_to_biomass_multiplier
clay_pct = float(D_site.attrs["clay_fraction"])

example_site = pd.DataFrame({
    "time": D_site.time.data,
    "temp_c": D_site.temperature_celcius[:, 0, 0].data,
    "precip_mm": D_site.precipitation_mm[:, 0, 0].data,
    "biomass_gC_m2": biomass,
})

out_site = simulate_soil_water_balance(
    example_site,
    date_col="time",
    elevation_m=elevation,
    clay_pct=clay_pct
)
out_site["time"] = pd.to_datetime(out_site["time"])